### Data ingestion -> VectorDB

In [132]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [133]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: RAG_Test_Document.pdf
  ✓ Loaded 1 pages

Total documents loaded: 1


In [134]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-30T18:20:05+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-30T18:20:05+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\RAG_Test_Document.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'RAG_Test_Document.pdf', 'file_type': 'pdf'}, page_content='RAG Test Document\nThis document is intentionally created to test a Retrieval-Augmented Generation (RAG) pipeline.\nEmployee Information\nEmployee Name: Alice Johnson\nEmployee ID: EMP-2048\nDepartment: Artificial Intelligence\nManager: David Miller\nOffice Location: Mumbai\nJoining Date: 12 January 2024\nProject Information\nCurrent Project: DocQuery AI\nTech Stack: FastAPI, React, ChromaDB, LangChain, Sentence Transformers, Groq Llama 3.3.\nThe application uses Retrieval-Augmented Generation (RAG) to answer questions fro

In [135]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [136]:
chunks=split_documents(all_pdf_documents)
chunks

Split 1 documents into 1 chunks

Example chunk:
Content: RAG Test Document
This document is intentionally created to test a Retrieval-Augmented Generation (RAG) pipeline.
Employee Information
Employee Name: Alice Johnson
Employee ID: EMP-2048
Department: Ar...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-30T18:20:05+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-30T18:20:05+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\RAG_Test_Document.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'RAG_Test_Document.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-30T18:20:05+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-30T18:20:05+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\RAG_Test_Document.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'RAG_Test_Document.pdf', 'file_type': 'pdf'}, page_content='RAG Test Document\nThis document is intentionally created to test a Retrieval-Augmented Generation (RAG) pipeline.\nEmployee Information\nEmployee Name: Alice Johnson\nEmployee ID: EMP-2048\nDepartment: Artificial Intelligence\nManager: David Miller\nOffice Location: Mumbai\nJoining Date: 12 January 2024\nProject Information\nCurrent Project: DocQuery AI\nTech Stack: FastAPI, React, ChromaDB, LangChain, Sentence Transformers, Groq Llama 3.3.\nThe application uses Retrieval-Augmented Generation (RAG) to answer questions fro

### Embedding, VectorDB

In [137]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [138]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6171.53it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\PRATEEKK\AppData\Local\Temp\ipykernel_1584\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [139]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 1


In [140]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-30T18:20:05+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-30T18:20:05+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\RAG_Test_Document.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'RAG_Test_Document.pdf', 'file_type': 'pdf'}, page_content='RAG Test Document\nThis document is intentionally created to test a Retrieval-Augmented Generation (RAG) pipeline.\nEmployee Information\nEmployee Name: Alice Johnson\nEmployee ID: EMP-2048\nDepartment: Artificial Intelligence\nManager: David Miller\nOffice Location: Mumbai\nJoining Date: 12 January 2024\nProject Information\nCurrent Project: DocQuery AI\nTech Stack: FastAPI, React, ChromaDB, LangChain, Sentence Transformers, Groq Llama 3.3.\nThe application uses Retrieval-Augmented Generation (RAG) to answer questions fro

In [141]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.76it/s]

Generated embeddings with shape: (1, 384)
Adding 1 documents to vector store...
Successfully added 1 documents to vector store
Total documents in collection: 2


### Retrieval pipeline (VectorDB)

In [142]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):

                    print(f"Distance: {distance}")

                retrieved_docs.append({
                    'id': doc_id,
                    'content': document,
                    'metadata': metadata,
                    'distance': distance,
                    'rank': i + 1
                })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [143]:
rag_retriever

In [144]:
rag_retriever.retrieve("Employee name")

Retrieving documents for query: 'Employee name'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 76.79it/s]

Generated embeddings with shape: (1, 384)
Distance: 1.2447690963745117
Distance: 1.2447690963745117
Retrieved 1 documents (after filtering)


[{'id': 'doc_7e98c64c_0',
  'content': 'RAG Test Document\nThis document is intentionally created to test a Retrieval-Augmented Generation (RAG) pipeline.\nEmployee Information\nEmployee Name: Alice Johnson\nEmployee ID: EMP-2048\nDepartment: Artificial Intelligence\nManager: David Miller\nOffice Location: Mumbai\nJoining Date: 12 January 2024\nProject Information\nCurrent Project: DocQuery AI\nTech Stack: FastAPI, React, ChromaDB, LangChain, Sentence Transformers, Groq Llama 3.3.\nThe application uses Retrieval-Augmented Generation (RAG) to answer questions from uploaded\nPDF documents.\nRandom Facts\nThe company cafeteria serves pasta every Wednesday.\nThe office Wi-Fi password is AI@2026Lab.',
  'metadata': {'subject': '(unspecified)',
   'moddate': '2026-06-30T18:20:05+00:00',
   'source': '..\\data\\pdf\\RAG_Test_Document.pdf',
   'author': '(anonymous)',
   'source_file': 'RAG_Test_Document.pdf',
   'page_label': '1',
   'producer': 'ReportLab PDF Library - (opensource)',
   'cre

In [145]:
rag_retriever.retrieve("End Semester Theory Examination")

Retrieving documents for query: 'End Semester Theory Examination'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 70.61it/s]

Generated embeddings with shape: (1, 384)
Distance: 1.708359956741333
Distance: 1.708359956741333
Retrieved 1 documents (after filtering)


[{'id': 'doc_7e98c64c_0',
  'content': 'RAG Test Document\nThis document is intentionally created to test a Retrieval-Augmented Generation (RAG) pipeline.\nEmployee Information\nEmployee Name: Alice Johnson\nEmployee ID: EMP-2048\nDepartment: Artificial Intelligence\nManager: David Miller\nOffice Location: Mumbai\nJoining Date: 12 January 2024\nProject Information\nCurrent Project: DocQuery AI\nTech Stack: FastAPI, React, ChromaDB, LangChain, Sentence Transformers, Groq Llama 3.3.\nThe application uses Retrieval-Augmented Generation (RAG) to answer questions from uploaded\nPDF documents.\nRandom Facts\nThe company cafeteria serves pasta every Wednesday.\nThe office Wi-Fi password is AI@2026Lab.',
  'metadata': {'title': '(anonymous)',
   'producer': 'ReportLab PDF Library - (opensource)',
   'keywords': '',
   'author': '(anonymous)',
   'total_pages': 1,
   'file_type': 'pdf',
   'source': '..\\data\\pdf\\RAG_Test_Document.pdf',
   'doc_index': 0,
   'trapped': '/False',
   'creator':

### Augmentation + Generation

In [146]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [147]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [148]:
class GroqLLM:
    def __init__(self, model_name: str = "llama-3.3-70b-versatile", api_key: str = None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [149]:
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: llama-3.3-70b-versatile
Groq LLM initialized successfully!


### Vectordb Context pipeline With LLM output

In [150]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=1024
)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [151]:
print("Total documents in vector store:", vectorstore.collection.count())

Total documents in vector store: 2


In [152]:
data = vectorstore.collection.get()

for doc in data["documents"]:
    print(doc)

RAG Test Document
This document is intentionally created to test a Retrieval-Augmented Generation (RAG) pipeline.
Employee Information
Employee Name: Alice Johnson
Employee ID: EMP-2048
Department: Artificial Intelligence
Manager: David Miller
Office Location: Mumbai
Joining Date: 12 January 2024
Project Information
Current Project: DocQuery AI
Tech Stack: FastAPI, React, ChromaDB, LangChain, Sentence Transformers, Groq Llama 3.3.
The application uses Retrieval-Augmented Generation (RAG) to answer questions from uploaded
PDF documents.
Random Facts
The company cafeteria serves pasta every Wednesday.
The office Wi-Fi password is AI@2026Lab.
RAG Test Document
This document is intentionally created to test a Retrieval-Augmented Generation (RAG) pipeline.
Employee Information
Employee Name: Alice Johnson
Employee ID: EMP-2048
Department: Artificial Intelligence
Manager: David Miller
Office Location: Mumbai
Joining Date: 12 January 2024
Project Information
Current Project: DocQuery AI
Tech 

In [153]:
answer=rag_simple("Who is Alice Johnson?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'Who is Alice Johnson?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 18.34it/s]

Generated embeddings with shape: (1, 384)
Distance: 1.6228303909301758
Distance: 1.6228303909301758
Retrieved 1 documents (after filtering)


Alice Johnson is an employee in the Artificial Intelligence department with ID EMP-2048.


In [154]:
answer=rag_simple("What is name of the employee?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is name of the employee?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.42it/s]

Generated embeddings with shape: (1, 384)
Distance: 1.300217866897583
Distance: 1.300217866897583
Retrieved 1 documents (after filtering)
Alice Johnson